In [17]:
import subprocess
from pydub import AudioSegment
import math


def videoToAudio(video_url, audio_url):
    command = ["ffmpeg", "-i", video_url, "-vn", audio_url]
    subprocess.run(command)


def chunkAudio(audio_url, chunk_size, chunk_url):
    audio = AudioSegment.from_mp3(audio_url)
    chunk_len = chunk_size * 60 * 1000

    for i in range(math.ceil(len(audio) / chunk_len)):
        chunk = audio[i * chunk_len : (i + 1) * chunk_len]
        chunk.export(f"{chunk_url}/audio_chunk_{i}.mp3")

In [18]:
videoToAudio("./files/video/video1.mp4", "./files/audio/audio1.mp3")

ffmpeg version 9.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.4.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/9.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --disable-libxcb --disable-xlib --enable-neon
  libavutil      61. 19.100 / 61. 19.100
  libavcodec     63. 24.100 / 63. 24.100
  libavformat    63. 16.100 / 63. 16.100
  libavdevice    63.  5.100 / 63.  5.100
  libavfilter    12.  8.100 / 12.  8.100
  libswscale     10.  5.100 / 10.  5.100
  libswresample   7.  6.100 /  7.  6.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from './files/video/video1.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isom

In [19]:
chunkAudio("./files/audio/audio1.mp3", 5, "./files/chunk")

In [32]:
import openai
import glob


def transcript_audio(audio_url, destination):
    audios = sorted(glob.glob(f"{audio_url}/*.mp3"))
    script_text = ""
    for audio in audios:
        with open(audio, "rb") as upload_file:
            script = openai.audio.transcriptions.create(
                model="gpt-transcribe", file=upload_file
            )
            script_text += script.text
    with open(destination, "w") as file:
        file.write(script_text)


transcript_audio("./files/chunk", "./files/transcript.txt")